### Setup: Drive, librerías y constantes del proyecto

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import numpy as np
import pandas as pd


CARPETA       = "/content/drive/MyDrive/Archivos_CSV_Deteccion_Fraude"
TAMANO_TRAIN  = 100_000        # filas del train reducido (estratificado)
SPLIT_RATIO   = 0.80           # 80% train / 20% test temporal
RANDOM_STATE  = 42

# --- Verificacion de archivos -----
print("Archivos en la carpeta:")
for f in sorted(os.listdir(CARPETA)):
    ruta = os.path.join(CARPETA, f)
    print(f"  {f:35s}  {os.path.getsize(ruta)/1024**2:8.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivos en la carpeta:
  test_df_etapa_inicial.parquet            16.2 MB
  train_df_etapa_inicial.parquet           13.7 MB
  train_identity.csv                       25.3 MB
  train_transaction.csv                   651.7 MB


### Carga y optimización de train_transaction

In [2]:
train_transaction = pd.read_csv(
    os.path.join(CARPETA, "train_transaction.csv"),
    low_memory=False
)

# Downcast de tipos para ahorrar memoria
for col in train_transaction.select_dtypes(include=["float64"]).columns:
    train_transaction[col] = train_transaction[col].astype("float32")

for col in train_transaction.select_dtypes(include=["int64"]).columns:
    train_transaction[col] = pd.to_numeric(train_transaction[col], downcast="integer")

for col in train_transaction.select_dtypes(include=["object"]).columns:
    train_transaction[col] = train_transaction[col].astype("category")

# Reporte
print(f"Dimensiones : {train_transaction.shape}")
print(f"Memoria     : {train_transaction.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nDistribución isFraud:")
print(train_transaction['isFraud'].value_counts().to_string())
print(f"% fraude    : {train_transaction['isFraud'].mean()*100:.4f}%")

Dimensiones : (590540, 394)
Memoria     : 861.1 MB

Distribución isFraud:
isFraud
0    569877
1     20663
% fraude    : 3.4990%


### Orden temporal, split principal y submuestreo estratificado del train

In [3]:

# Orden cronologico
train_transaction = (
    train_transaction
    .sort_values("TransactionDT")
    .reset_index(drop=True)
)

# Split temporal 80/20
split_idx = int(len(train_transaction) * SPLIT_RATIO)
tx_train  = train_transaction.iloc[:split_idx].copy()
tx_test   = train_transaction.iloc[split_idx:].copy()

print("--- Split temporal ---")
print(f"tx_train : {tx_train.shape}   fraude: {tx_train['isFraud'].mean()*100:.4f}%")
print(f"tx_test  : {tx_test.shape}    fraude: {tx_test['isFraud'].mean()*100:.4f}%")

# Submuestreo estratificado SOLO en train (preservando proporción de fraude)
idx_fr = tx_train[tx_train["isFraud"] == 1].index
idx_nf = tx_train[tx_train["isFraud"] == 0].index

n_fr = int(TAMANO_TRAIN * tx_train["isFraud"].mean())
n_nf = TAMANO_TRAIN - n_fr

rng = np.random.RandomState(RANDOM_STATE)
sel_idx = np.sort(np.concatenate([
    rng.choice(idx_fr, n_fr, replace=False),
    rng.choice(idx_nf, n_nf, replace=False),
]))

tx_train_small = tx_train.iloc[sel_idx].reset_index(drop=True)

print("\n--- Train reducido --")
print(f"tx_train_small : {tx_train_small.shape}")
print(f"fraude         : {tx_train_small['isFraud'].mean()*100:.4f}%")

# Liberar el dataset completo (ya no se necesita)
del train_transaction, tx_train
gc.collect()

--- Split temporal ---
tx_train : (472432, 394)   fraude: 3.5135%
tx_test  : (118108, 394)    fraude: 3.4409%

--- Train reducido ---
tx_train_small : (100000, 394)
fraude         : 3.5130%


0

###  Carga y optimización de train_identity

In [4]:
train_identity = pd.read_csv(
    os.path.join(CARPETA, "train_identity.csv"),
    low_memory=False
)

for col in train_identity.select_dtypes(include=["float64"]).columns:
    train_identity[col] = train_identity[col].astype("float32")

for col in train_identity.select_dtypes(include=["int64"]).columns:
    train_identity[col] = pd.to_numeric(train_identity[col], downcast="integer")

for col in train_identity.select_dtypes(include=["object"]).columns:
    train_identity[col] = train_identity[col].astype("category")

print(f"Dimensiones : {train_identity.shape}")
print(f"Memoria     : {train_identity.memory_usage(deep=True).sum()/1024**2:.1f} MB")

Dimensiones : (144233, 41)
Memoria     : 16.2 MB


### Merge final, verificación de cobertura y limpieza

In [5]:
# LEFT JOIN identity sobre train reducido y sobre test completo
train_df = tx_train_small.merge(train_identity, on="TransactionID", how="left")
test_df  = tx_test.merge(train_identity,        on="TransactionID", how="left")

# Verificacion de cobertura real de identity (post-merge)
cov_train = train_df['DeviceType'].notna().mean() * 100
cov_test  = test_df['DeviceType'].notna().mean()  * 100

print("--- Dimensiones finales ---")
print(f"train_df : {train_df.shape}")
print(f"test_df  : {test_df.shape}")

print("\n--- Cobertura de identity (post-merge) ---")
print(f"train_df : {train_df['DeviceType'].notna().sum():>6,} / {len(train_df):>6,}  ({cov_train:.2f}%)")
print(f"test_df  : {test_df['DeviceType'].notna().sum():>6,} / {len(test_df):>6,}  ({cov_test:.2f}%)")

print("\n--- Memoria ---")
print(f"train_df : {train_df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"test_df  : {test_df.memory_usage(deep=True).sum()/1024**2:.1f} MB")

print("\n--- Tasa de fraude final ---")
print(f"train_df : {train_df['isFraud'].mean()*100:.4f}%")
print(f"test_df  : {test_df['isFraud'].mean()*100:.4f}%")

# Eliminar de memoria todo lo intermedio
del train_identity, tx_train_small, tx_test
gc.collect()

# Vista previaa
print("\n--- Primeras filas de train_df ---")
train_df.head()

--- Dimensiones finales ---
train_df : (100000, 434)
test_df  : (118108, 434)

--- Cobertura de identity (post-merge) ---
train_df : 24,956 / 100,000  (24.96%)
test_df  : 23,172 / 118,108  (19.62%)

--- Memoria ---
train_df : 156.7 MB
test_df  : 185.1 MB

--- Tasa de fraude final ---
train_df : 3.5130%
test_df  : 3.4409%

--- Primeras filas de train_df ---


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987002,0,86469,59.000000,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987006,0,86522,159.000000,W,12308,360.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987024,0,86821,73.949997,W,10112,360.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987026,0,86945,184.000000,W,17868,148.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987030,0,86994,35.000000,W,13276,555.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Separación inicial para modelado

In [6]:
# Referencia temporal
train_dt_ref = train_df["TransactionDT"].copy()
test_dt_ref  = test_df["TransactionDT"].copy()

# Variables objetivo
y_train = train_df["isFraud"].copy()
y_test  = test_df["isFraud"].copy()

# Variables predictoras
X_train = train_df.drop(columns=["TransactionID", "TransactionDT", "isFraud"]).copy()
X_test  = test_df.drop(columns=["TransactionID", "TransactionDT", "isFraud"]).copy()

print("--- Dimensiones para modelado ---")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train fraude: {y_train.mean()*100:.4f}%")
print(f"y_test  fraude: {y_test.mean()*100:.4f}%")

--- Dimensiones para modelado ---
X_train: (100000, 431)
X_test : (118108, 431)
y_train fraude: 3.5130%
y_test  fraude: 3.4409%


### Eliminación de columnas con demasiados valores nulos

In [7]:
missing_ratio = X_train.isnull().mean().sort_values(ascending=False)
cols_nulos_altos = missing_ratio[missing_ratio > 0.90].index.tolist()

X_train = X_train.drop(columns=cols_nulos_altos)
X_test  = X_test.drop(columns=cols_nulos_altos)

print("--- Eliminacion por cantidad de valores nulos altos ---")
print(f"Columnas eliminadas: {len(cols_nulos_altos)}")
print("Primeras columnas eliminadas:", cols_nulos_altos[:15])
print(f"Nuevo tamaño X_train: {X_train.shape}")
print(f"Nuevo tamaño X_test : {X_test.shape}")

--- Eliminación por nulos altos ---
Columnas eliminadas: 12
Primeras columnas eliminadas: ['id_24', 'id_25', 'id_08', 'id_07', 'id_21', 'id_27', 'id_22', 'id_26', 'id_23', 'D7', 'dist2', 'id_18']
Nuevo tamaño X_train: (100000, 419)
Nuevo tamaño X_test : (118108, 419)


### Identificación de tipos y eliminación de columnas altamente correlacionadas

In [8]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

corr_matrix = X_train[num_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
cols_correladas = [c for c in upper.columns if (upper[c] > 0.90).any()]

X_train = X_train.drop(columns=cols_correladas)
X_test  = X_test.drop(columns=cols_correladas)

# Actualizar listas de columnas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("--- Tipos de variables ---")
print(f"Numéricas   : {len(num_cols)}")
print(f"Categóricas : {len(cat_cols)}")

print("\n--- Eliminación por correlación alta ---")
print(f"Columnas eliminadas: {len(cols_correladas)}")
print("Primeras columnas eliminadas:", cols_correladas[:15])

print(f"\nTamaño final tras limpieza estructural:")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

# Liberar la matriz de correlación (puede pesar bastante)
del corr_matrix, upper
gc.collect()

--- Tipos de variables ---
Numéricas   : 218
Categóricas : 29

--- Eliminación por correlación alta ---
Columnas eliminadas: 172
Primeras columnas eliminadas: ['C2', 'C4', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C14', 'D2', 'D6', 'D12', 'V5', 'V11']

Tamaño final tras limpieza estructural:
X_train: (100000, 247)
X_test : (118108, 247)


0

###  Registro de decisiones tomadas

In [9]:
registro_preprocesamiento = {
    "columnas_eliminadas_nulos": cols_nulos_altos,
    "columnas_eliminadas_correlacion": cols_correladas,
    "n_columnas_finales_train": X_train.shape[1],
    "n_columnas_finales_test": X_test.shape[1],
    "n_numericas_finales": len(num_cols),
    "n_categoricas_finales": len(cat_cols),
}

print("--- Resumen de decisiones ---")
for k, v in registro_preprocesamiento.items():
    if isinstance(v, list):
        print(f"{k}: {len(v)} elementos")
    else:
        print(f"{k}: {v}")

--- Resumen de decisiones ---
columnas_eliminadas_nulos: 12 elementos
columnas_eliminadas_correlacion: 172 elementos
n_columnas_finales_train: 247
n_columnas_finales_test: 247
n_numericas_finales: 218
n_categoricas_finales: 29


### Imputación y clasificación por cardinalidad


In [10]:
# Imputacion de variiables numéricas con mediana de train
medianas_train = X_train[num_cols].median()
X_train[num_cols] = X_train[num_cols].fillna(medianas_train)
X_test[num_cols]  = X_test[num_cols].fillna(medianas_train)

# Imputacion de variables categoricas con etiqueta Missing
# (convertimos a object para evitar problemas con dtype category)
for col in cat_cols:
    X_train[col] = X_train[col].astype("object").fillna("Missing")
    X_test[col]  = X_test[col].astype("object").fillna("Missing")

# Clasificacion de categooricas por cardinalidad
low_card_cols  = [c for c in cat_cols if X_train[c].nunique() <= 10]
high_card_cols = [c for c in cat_cols if X_train[c].nunique() > 10]

print("--- Imputación completada ---")
print(f"Variables numéricas imputadas   : {len(num_cols)}")
print(f"Variables categóricas imputadas : {len(cat_cols)}")
print("\n--- Clasificación por cardinalidad ---")
print(f"Baja cardinalidad (<=10) : {len(low_card_cols)}")
print(f"Media/alta cardinalidad  : {len(high_card_cols)}")
print("\nPrimeras variables de baja cardinalidad:")
print(low_card_cols[:15])
print("\nPrimeras variables de media/alta cardinalidad:")
print(high_card_cols[:15])

--- Imputación completada ---
Variables numéricas imputadas   : 218
Variables categóricas imputadas : 29

--- Clasificación por cardinalidad ---
Baja cardinalidad (<=10) : 23
Media/alta cardinalidad  : 6

Primeras variables de baja cardinalidad:
['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16']

Primeras variables de media/alta cardinalidad:
['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo']


### Codificación categórica

In [11]:
# One-hot encoding para baja cardinalidad
X_train = pd.get_dummies(X_train, columns=low_card_cols, drop_first=False)
X_test  = pd.get_dummies(X_test,  columns=low_card_cols, drop_first=False)

# Frequency encoding para media/alta cardinalidad
for col in high_card_cols:
    freq_map = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq_map).astype("float32")
    X_test[col]  = X_test[col].map(freq_map).fillna(0).astype("float32")

# Alinear columnas entre train y test
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print("--- Codificación completada ---")
print(f"Dimensión final X_train: {X_train.shape}")
print(f"Dimensión final X_test : {X_test.shape}")

--- Codificación completada ---
Dimensión final X_train: (100000, 303)
Dimensión final X_test : (118108, 303)


### Verificación final del preprocesamiento



In [12]:
# Convertir columnas booleanas (generadas por get_dummies) a uint8
bool_cols = X_train.select_dtypes(include=["bool"]).columns.tolist()
if bool_cols:
    X_train[bool_cols] = X_train[bool_cols].astype("uint8")
    X_test[bool_cols]  = X_test[bool_cols].astype("uint8")

# Verificaciones
nulos_train = X_train.isnull().sum().sum()
nulos_test  = X_test.isnull().sum().sum()
inf_train   = np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum()
inf_test    = np.isinf(X_test.select_dtypes(include=[np.number])).sum().sum()

print("--- Verificación final ---")
print(f"Nulos en X_train: {nulos_train}")
print(f"Nulos en X_test : {nulos_test}")
print(f"Infinitos en X_train: {inf_train}")
print(f"Infinitos en X_test : {inf_test}")

print("\nDimensiones listas para modelado:")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

--- Verificación final ---
Nulos en X_train: 0
Nulos en X_test : 0
Infinitos en X_train: 0
Infinitos en X_test : 0

Dimensiones listas para modelado:
X_train: (100000, 303)
X_test : (118108, 303)
y_train: (100000,)
y_test : (118108,)


### Validación temporal interna dentro del train reducido

In [13]:
# Validación temporal interna
split_val_idx = int(len(X_train) * 0.85)

X_subtrain = X_train.iloc[:split_val_idx].copy()
X_val      = X_train.iloc[split_val_idx:].copy()
y_subtrain = y_train.iloc[:split_val_idx].copy()
y_val      = y_train.iloc[split_val_idx:].copy()

print("--- Validación temporal interna ---")
print(f"X_subtrain: {X_subtrain.shape}   fraude: {y_subtrain.mean()*100:.4f}%")
print(f"X_val     : {X_val.shape}        fraude: {y_val.mean()*100:.4f}%")

--- Validación temporal interna ---
X_subtrain: (85000, 303)   fraude: 3.5271%
X_val     : (15000, 303)        fraude: 3.4333%


###  Construcción de la rama manual y la rama referencial

In [14]:
# Rama referencial (sin escalado obligatorio)
X_subtrain_ref = X_subtrain.copy()
X_val_ref      = X_val.copy()
X_test_ref     = X_test.copy()

# Rama manual (se escalara)
X_subtrain_manual = X_subtrain.copy()
X_val_manual      = X_val.copy()
X_test_manual     = X_test.copy()

print("--- Ramas creadas ---")
print(f"Rama manual      - X_subtrain_manual: {X_subtrain_manual.shape}")
print(f"Rama referencial - X_subtrain_ref   : {X_subtrain_ref.shape}")

--- Ramas creadas ---
Rama manual      - X_subtrain_manual: (85000, 303)
Rama referencial - X_subtrain_ref   : (85000, 303)


### Escalado para la rama manual

In [15]:
# Detectar columnas numeericas en la rama manual
num_cols_manual = X_subtrain_manual.select_dtypes(include=[np.number]).columns.tolist()

# Media y desviacion estndar calculadas solo con subtrain
mean_manual = X_subtrain_manual[num_cols_manual].mean()
std_manual  = X_subtrain_manual[num_cols_manual].std().replace(0, 1)

# Escalado (forzando float32 para evitar duplicar el tamaño en la memoria)
X_subtrain_manual[num_cols_manual] = (
    (X_subtrain_manual[num_cols_manual] - mean_manual) / std_manual
).astype("float32")

X_val_manual[num_cols_manual] = (
    (X_val_manual[num_cols_manual] - mean_manual) / std_manual
).astype("float32")

X_test_manual[num_cols_manual] = (
    (X_test_manual[num_cols_manual] - mean_manual) / std_manual
).astype("float32")

print("--- Rama manual escalada ---")
print(f"X_subtrain_manual: {X_subtrain_manual.shape}")
print(f"X_val_manual     : {X_val_manual.shape}")
print(f"X_test_manual    : {X_test_manual.shape}")

--- Rama manual escalada ---
X_subtrain_manual: (85000, 303)
X_val_manual     : (15000, 303)
X_test_manual    : (118108, 303)


### Verificación final de las ramas


In [16]:
print("--- Verificación rama manual ---")
print(f"Nulos X_subtrain_manual: {X_subtrain_manual.isnull().sum().sum()}")
print(f"Nulos X_val_manual     : {X_val_manual.isnull().sum().sum()}")
print(f"Nulos X_test_manual    : {X_test_manual.isnull().sum().sum()}")

print("\n--- Verificación rama referencial ---")
print(f"Nulos X_subtrain_ref: {X_subtrain_ref.isnull().sum().sum()}")
print(f"Nulos X_val_ref     : {X_val_ref.isnull().sum().sum()}")
print(f"Nulos X_test_ref    : {X_test_ref.isnull().sum().sum()}")

print("\n--- Dimensiones finales ---")
print(f"Manual      - subtrain: {X_subtrain_manual.shape}, val: {X_val_manual.shape}, test: {X_test_manual.shape}")
print(f"Referencial - subtrain: {X_subtrain_ref.shape}, val: {X_val_ref.shape}, test: {X_test_ref.shape}")

--- Verificación rama manual ---
Nulos X_subtrain_manual: 0
Nulos X_val_manual     : 0
Nulos X_test_manual    : 0

--- Verificación rama referencial ---
Nulos X_subtrain_ref: 0
Nulos X_val_ref     : 0
Nulos X_test_ref    : 0

--- Dimensiones finales ---
Manual      - subtrain: (85000, 303), val: (15000, 303), test: (118108, 303)
Referencial - subtrain: (85000, 303), val: (15000, 303), test: (118108, 303)


### Funciones auxiliares para la comparación preliminar

In [17]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def predict_proba_lr(X, w, b):
    return sigmoid(X @ w + b)

def predict_lr(X, w, b, threshold=0.5):
    probs = predict_proba_lr(X, w, b)
    return (probs >= threshold).astype(np.uint8)

def confusion_matrix_binary(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def precision_recall_f1_mcc(y_true, y_pred):
    tp, tn, fp, fn = confusion_matrix_binary(y_true, y_pred)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

def auc_roc_manual(y_true, y_prob):
    order = np.argsort(-y_prob)
    y_sorted = y_true[order]

    P = np.sum(y_true == 1)
    N = np.sum(y_true == 0)

    if P == 0 or N == 0:
        return 0.0

    tp = 0
    fp = 0
    tpr = [0.0]
    fpr = [0.0]

    for yi in y_sorted:
        if yi == 1:
            tp += 1
        else:
            fp += 1
        tpr.append(tp / P)
        fpr.append(fp / N)

    tpr.append(1.0)
    fpr.append(1.0)

    return np.trapz(tpr, fpr)

def auc_pr_manual(y_true, y_prob):
    order = np.argsort(-y_prob)
    y_sorted = y_true[order]

    P = np.sum(y_true == 1)
    if P == 0:
        return 0.0

    tp = 0
    fp = 0
    recall_vals = [0.0]
    precision_vals = [1.0]

    for yi in y_sorted:
        if yi == 1:
            tp += 1
        else:
            fp += 1

        recall = tp / P
        precision = tp / (tp + fp)

        recall_vals.append(recall)
        precision_vals.append(precision)

    return np.trapz(precision_vals, recall_vals)

###  Regresión Logística manual básica

In [18]:
def train_logistic_basic(X_train, y_train, alpha=0.01, lambda_reg=0.01, epochs=200):
    X_np = X_train.to_numpy(dtype=np.float32)
    y_np = y_train.to_numpy(dtype=np.float32)
    n_samples, n_features = X_np.shape

    w = np.zeros(n_features, dtype=np.float32)
    b = 0.0

    n1 = np.sum(y_np == 1)
    n0 = np.sum(y_np == 0)
    n  = len(y_np)
    w1 = n / (2 * n1)
    w0 = n / (2 * n0)

    for epoch in range(epochs):
        y_prob = predict_proba_lr(X_np, w, b)
        sample_weights = np.where(y_np == 1, w1, w0).astype(np.float32)
        error = (y_prob - y_np) * sample_weights
        dw = (X_np.T @ error) / n + (lambda_reg / n) * w
        db = np.mean(error)
        w -= alpha * dw
        b -= alpha * db

    return w, b

alpha_lr   = 0.01
lambda_lr  = 0.01
epochs_lr  = 200

w_lr, b_lr = train_logistic_basic(
    X_subtrain_manual, y_subtrain,
    alpha=alpha_lr, lambda_reg=lambda_lr, epochs=epochs_lr
)

lr_probs = predict_proba_lr(X_test_manual.to_numpy(dtype=np.float32), w_lr, b_lr)
lr_pred  = (lr_probs >= 0.5).astype(np.uint8)

y_test_np = y_test.to_numpy()
lr_metrics = precision_recall_f1_mcc(y_test_np, lr_pred)
lr_auc_roc = auc_roc_manual(y_test_np, lr_probs)
lr_auprc   = auc_pr_manual(y_test_np, lr_probs)

print("--- Regresión Logística manual ---")
print(f"Precision: {lr_metrics['precision']:.4f}")
print(f"Recall   : {lr_metrics['recall']:.4f}")
print(f"F1-score : {lr_metrics['f1']:.4f}")
print(f"MCC      : {lr_metrics['mcc']:.4f}")
print(f"AUC-ROC  : {lr_auc_roc:.4f}")
print(f"AUPRC    : {lr_auprc:.4f}")

--- Regresión Logística manual ---
Precision: 0.0886
Recall   : 0.7832
F1-score : 0.1593
MCC      : 0.1967
AUC-ROC  : 0.8221
AUPRC    : 0.1629


/tmp/ipykernel_3852/2165165759.py:74: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr, fpr)
/tmp/ipykernel_3852/2165165759.py:101: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(precision_vals, recall_vals)


### KNN manual

In [19]:
def knn_predict_proba_chunked(X_train, y_train, X_test, k=5, chunk_size=500):
    Xtr = X_train.to_numpy(dtype=np.float32)
    ytr = y_train.to_numpy(dtype=np.uint8)
    Xte = X_test.to_numpy(dtype=np.float32)

    train_norm = np.sum(Xtr ** 2, axis=1)[None, :]
    probs = []

    for start in range(0, len(Xte), chunk_size):
        end = min(start + chunk_size, len(Xte))
        X_chunk = Xte[start:end]

        # ||x - y||^2 = ||x||^2 + ||y||^2 - 2 * x·y
        chunk_norm = np.sum(X_chunk ** 2, axis=1)[:, None]
        cross      = X_chunk @ Xtr.T
        dists_sq   = np.maximum(chunk_norm + train_norm - 2 * cross, 0)

        idx_neighbors  = np.argpartition(dists_sq, kth=k-1, axis=1)[:, :k]
        neighbor_labels = ytr[idx_neighbors]
        chunk_probs    = neighbor_labels.mean(axis=1)
        probs.append(chunk_probs)

    return np.concatenate(probs)

k_knn = 5
knn_probs = knn_predict_proba_chunked(
    X_subtrain_manual, y_subtrain, X_test_manual,
    k=k_knn, chunk_size=500
)
knn_pred = (knn_probs >= 0.5).astype(np.uint8)

knn_metrics = precision_recall_f1_mcc(y_test_np, knn_pred)
knn_auc_roc = auc_roc_manual(y_test_np, knn_probs)
knn_auprc   = auc_pr_manual(y_test_np, knn_probs)

print("--- KNN manual ---")
print(f"Precision: {knn_metrics['precision']:.4f}")
print(f"Recall   : {knn_metrics['recall']:.4f}")
print(f"F1-score : {knn_metrics['f1']:.4f}")
print(f"MCC      : {knn_metrics['mcc']:.4f}")
print(f"AUC-ROC  : {knn_auc_roc:.4f}")
print(f"AUPRC    : {knn_auprc:.4f}")

--- KNN manual ---
Precision: 0.7207
Recall   : 0.1905
F1-score : 0.3013
MCC      : 0.3607
AUC-ROC  : 0.6682
AUPRC    : 0.2708


/tmp/ipykernel_3852/2165165759.py:74: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr, fpr)
/tmp/ipykernel_3852/2165165759.py:101: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(precision_vals, recall_vals)


###  Árbol de Decisión y Random Forest referenciales

In [20]:
dt_model = DecisionTreeClassifier(
    class_weight="balanced", random_state=42, max_depth=8
)
rf_model = RandomForestClassifier(
    n_estimators=50, max_depth=10, class_weight="balanced",
    random_state=42, n_jobs=-1
)

# Arbol de Decision
dt_model.fit(X_subtrain_ref, y_subtrain)
dt_probs = dt_model.predict_proba(X_test_ref)[:, 1]
dt_pred  = (dt_probs >= 0.5).astype(np.uint8)

dt_metrics = precision_recall_f1_mcc(y_test_np, dt_pred)
dt_auc_roc = auc_roc_manual(y_test_np, dt_probs)
dt_auprc   = auc_pr_manual(y_test_np, dt_probs)

print("--- Árbol de Decisión referencial ---")
print(f"Precision: {dt_metrics['precision']:.4f}")
print(f"Recall   : {dt_metrics['recall']:.4f}")
print(f"F1-score : {dt_metrics['f1']:.4f}")
print(f"MCC      : {dt_metrics['mcc']:.4f}")
print(f"AUC-ROC  : {dt_auc_roc:.4f}")
print(f"AUPRC    : {dt_auprc:.4f}")

# Random Forest
rf_model.fit(X_subtrain_ref, y_subtrain)
rf_probs = rf_model.predict_proba(X_test_ref)[:, 1]
rf_pred  = (rf_probs >= 0.5).astype(np.uint8)

rf_metrics = precision_recall_f1_mcc(y_test_np, rf_pred)
rf_auc_roc = auc_roc_manual(y_test_np, rf_probs)
rf_auprc   = auc_pr_manual(y_test_np, rf_probs)

print("\n--- Random Forest referencial ---")
print(f"Precision: {rf_metrics['precision']:.4f}")
print(f"Recall   : {rf_metrics['recall']:.4f}")
print(f"F1-score : {rf_metrics['f1']:.4f}")
print(f"MCC      : {rf_metrics['mcc']:.4f}")
print(f"AUC-ROC  : {rf_auc_roc:.4f}")
print(f"AUPRC    : {rf_auprc:.4f}")

/tmp/ipykernel_3852/2165165759.py:74: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr, fpr)
/tmp/ipykernel_3852/2165165759.py:101: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(precision_vals, recall_vals)


--- Árbol de Decisión referencial ---
Precision: 0.1278
Recall   : 0.6329
F1-score : 0.2126
MCC      : 0.2322
AUC-ROC  : 0.7702
AUPRC    : 0.3182

--- Random Forest referencial ---
Precision: 0.2264
Recall   : 0.5433
F1-score : 0.3196
MCC      : 0.3160
AUC-ROC  : 0.8537
AUPRC    : 0.3871


/tmp/ipykernel_3852/2165165759.py:74: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tpr, fpr)
/tmp/ipykernel_3852/2165165759.py:101: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(precision_vals, recall_vals)


###  Tabla comparativa preliminar


In [21]:
resultados_preliminares = pd.DataFrame([
    {
        "Modelo": "Regresión Logística (manual)",
        "Precision": lr_metrics["precision"],
        "Recall": lr_metrics["recall"],
        "F1": lr_metrics["f1"],
        "MCC": lr_metrics["mcc"],
        "AUC_ROC": lr_auc_roc,
        "AUPRC": lr_auprc
    },
    {
        "Modelo": "KNN (manual)",
        "Precision": knn_metrics["precision"],
        "Recall": knn_metrics["recall"],
        "F1": knn_metrics["f1"],
        "MCC": knn_metrics["mcc"],
        "AUC_ROC": knn_auc_roc,
        "AUPRC": knn_auprc
    },
    {
        "Modelo": "Árbol de Decisión (referencial)",
        "Precision": dt_metrics["precision"],
        "Recall": dt_metrics["recall"],
        "F1": dt_metrics["f1"],
        "MCC": dt_metrics["mcc"],
        "AUC_ROC": dt_auc_roc,
        "AUPRC": dt_auprc
    },
    {
        "Modelo": "Random Forest (referencial)",
        "Precision": rf_metrics["precision"],
        "Recall": rf_metrics["recall"],
        "F1": rf_metrics["f1"],
        "MCC": rf_metrics["mcc"],
        "AUC_ROC": rf_auc_roc,
        "AUPRC": rf_auprc
    }
])

resultados_preliminares = resultados_preliminares.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

resultados_preliminares


,Modelo,Precision,Recall,F1,MCC,AUC_ROC,AUPRC
0,Random Forest (referencial),0.226369,0.543307,0.319583,0.315969,0.853701,0.387119
1,KNN (manual),0.720670,0.190453,0.301285,0.360663,0.668190,0.270781
2,Árbol de Decisión (referencial),0.127769,0.632874,0.212615,0.232159,0.770237,0.318173
3,Regresión Logística (manual),0.088646,0.783219,0.159265,0.196657,0.822067,0.162858


### Resumen rápido del comportamiento observado


In [22]:
mejor_f1 = resultados_preliminares.iloc[0]
mejor_recall = resultados_preliminares.sort_values(by="Recall", ascending=False).iloc[0]
mejor_precision = resultados_preliminares.sort_values(by="Precision", ascending=False).iloc[0]

print("--- Resumen preliminar ---")
print(f"Mejor F1-score : {mejor_f1['Modelo']} ({mejor_f1['F1']:.4f})")
print(f"Mejor Recall   : {mejor_recall['Modelo']} ({mejor_recall['Recall']:.4f})")
print(f"Mejor Precision: {mejor_precision['Modelo']} ({mejor_precision['Precision']:.4f})")


--- Resumen preliminar ---
Mejor F1-score : Random Forest (referencial) (0.3196)
Mejor Recall   : Regresión Logística (manual) (0.7832)
Mejor Precision: KNN (manual) (0.7207)
